# NB1: Working with Observational Climate Data using climate4R
## 1. Introduction

This notebook provides an introduction to the analysis of observational climate datasets using the climate4R framework in R. Throughout the notebook, we will work with daily maximum temperature observations collected from meteorological stations in Pakistan. The main objective is to illustrate a typical exploratory analysis workflow, including the computation of climatological statistics, the characterization of temporal variability, and the comparison of local observations with gridded reanalysis products.

Unlike a conventional R script, this notebook has been designed as a learning resource. Each section introduces the scientific motivation behind the analysis before presenting the corresponding code. 

### 1.1 Learning Objectives

After completing this notebook, you should be able to:

* Understand the structure of climate4R Grid objects.
* Compute basic climatological statistics.
* Analyse seasonal and interannual variability.
* Produce spatial maps of climatological indicators.
* Analyse time series from individual weather stations.
* Compare station observations with ERA5 reanalysis data.
* Interpret common diagnostics used in climate data evaluation.

## 2. Preparing the Working Environment

First, it is necessary to define a few parameters controlling the analysis to be undertaken and load the required libraries.

The climate4R ecosystem provides a collection of packages specifically designed for handling climate data. These packages  (`loadeR`, `transformeR`, `visualizeR`) allow users to manipulate both station observations and gridded datasets through a common data structure, making subsequent analyses (including temporal and spatial subsetting, regridding, visualization, bias correction and statistical downscaling) considerably easier.

Some extra libraries outside climate4R (`RColorBrewer`, `lattice`, `gridExtra`) are also loaded to facilitate the production of publication-quality figures.

In [ ]:
rm(list = ls())

## params ##
dirbase = "/home/jovyan"
dirdata.obs = sprintf("%s/data/obs", dirbase)
dirdata.era5 = "https://thredds.climate.ifca.es/thredds/dodsC/fao/trainings/pakistan-202608/data/era5"

#lon = c(60, 80)
#lat = c(22, 38)

lon = c(66, 75)
lat = c(24, 35)

## libraries ##
library(loadeR)
library(transformeR)
library(visualizeR)
library(RColorBrewer)
library(lattice)
library(gridExtra)

## auxiliary functions ##
source(sprintf("%s/notebooks/auxiliary_functions.R", dirbase))

## 2. Loading the Observational Dataset

The first step consists of loading the observational dataset containing daily maximum temperature (Tmax).

The dataset is stored as an R binary object (.rda) and follows the climate4R Grid structure. Although the observations originate from meteorological stations (i.e. point locations), they are stored in the same data model structure dedicated to gridded datasets in climate4R. This unified representation allows the same functions to operate seamlessly on both station and gridded datasets.

After loading the dataset, we rename the object to obs, which will be used throughout the remainder of the notebook.

In [ ]:
## loading observed tmax ##
load(sprintf("%s/tmax.rda", dirdata.obs))
obs = data; remove(data)

## 3. Exploring the *Grid* object

Before carrying out any statistical analysis, it is good practice to inspect the structure of the dataset.

A climate4R Grid object contains much more than the climate variable itself. Besides the data values, it stores metadata describing the variable, temporal information, geographical coordinates and several additional attributes required by subsequent functions.

Here, we examine these different components to better understand how observational datasets are organised within the climate4R framework.

In [ ]:
## C4R grid's structure (for point-wise data) ##
getShape(***)
***$Metadata
getTimeResolution(***)
head(***$Dates$start)
tail(***$Dates$start)
range(***$Dates)
***$Variable
***$xyCoords

## 4. Computing the Mean Climatology

One of the first analyses performed on any climate dataset is the computation of the climatological mean. A climatology represents the average state of a climate variable over a sufficiently long period and provides a concise description of its typical spatial distribution.

For maximum temperature, the climatological mean highlights large-scale geographical patterns that are controlled by factors such as latitude, elevation, continentality and proximity to the sea. Before investigating variability or long-term trends, it is useful to identify these dominant spatial features.

The `climatology()` function computes the temporal average at every observation point, while `spatialPlot()` displays the resulting field over the study area.

In [ ]:
## computing mean climatology ##
clim.obs = climatology(***)
getShape(clim.obs)

In [ ]:
## plotting mean climatology ##
spatialPlot(***, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "Mean climatology")

Several optional input arguments may be passed to `spatialPlot()` to improve the aesthetics of the figure.

In [ ]:
## improving figure's aesthetics ##
cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(18, 38, 1)
spatialPlot(***, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "Mean climatology",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins,
                                                                              cex = 1,
                                                                              labels = cb.bins))))))

## 5. Characterising Temperature Extremes

Average conditions provide only part of the information contained in a climate dataset. Many practical applications (e.g. agriculture, hydrology and risk assessment) are more sensitive to extreme events than to average conditions.

A common way of describing extremes is through percentiles. Instead of focusing exclusively on the absolute minimum and maximum values, percentiles provide robust estimates of unusually low and unusually high temperatures.

Next, we compute:

* the 2nd percentile, representing very low maximum temperature values;
* the 98th percentile, representing very high maximum temperature values (these are particulary important).

These statistics summarize the tails of the temperature distribution while remaining relatively insensitive to isolated outliers. 

They are next computed using the `climatology()` function. Afterwards, `makeMultiGrid()` is used to plot the two resulting maps side by side, facilitating thus direct comparison.

In [ ]:
## compute (and plot) extreme percentiles ##
p2 = climatology(obs, clim.fun = list(***))
p98 = climatology(obs, clim.fun = list(***))

In [ ]:
## plotting ##
cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(10, 50, 2)
spatialPlot(***, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(2, 1),
            main = "Extreme percentiles",
            names.attr = c("P2", "P98"),
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins,
                                                                              cex = 1,
                                                                              labels = cb.bins))))))

## 6. Estimating Interannual Trends

Climate variability occurs over many temporal scales. Beyond day-to-day fluctuations and the seasonal cycle, climate variables may also exhibit long-term changes.

To investigate these changes, daily observations are first aggregated into annual means (`aggregateGrid()` function). Averaging over an entire year substantially reduces short-term variability and allows long-term tendencies to become more apparent.

The trend is then estimated independently at every station calling the `computeTrend()` function from `climatology()`.

Plotting the spatial distribution of the trends provides valuable information about regional coherence. A diverging colour scale is adopted so that:

* negative trends appear in blue;
* positive trends appear in red.

This representation facilitates the identification of regions undergoing warming or cooling.

**Note:** Remember that a trend estimated from observations is influenced by the length of the available record and by natural climate variability. A positive trend does not necessarily imply statistically significant climate change; it simply indicates the direction and magnitude of the estimated linear tendency over the analysed period.

In [ ]:
## compute (and plot) interannual trends ##
obs.y = aggregateGrid(obs, ***)
trend.y = climatology(obs.y, clim.fun = list(***))

In [ ]:
## plotting ##
cols = rev(brewer.pal(9, "RdBu"))
cb.bins = seq(-0.1, 0.1, 0.01)
spatialPlot(***, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "Yearly trends",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins,
                                                                              cex = 1,
                                                                              labels = cb.bins))))))

## 7. Seasonal Analysis (for DJF)

The climate of most regions changes substantially throughout the year. Consequently, many climatological indicators differ markedly between seasons.

For example, maximum temperatures during winter are controlled by atmospheric processes that differ from those dominating summer conditions. Analysing the entire year at once may therefore mask important seasonal features.

To illustrate this idea, the notebook first focuses on the December-January-February (DJF) season (`subsetGrid()` function).

For DJF, we compute:

* the mean climatology;
* the standard deviation (of daily values);
* the annual trend;
* the 98th percentile (of daily values).

We first plot a map for each of the four diagnostics, which provide complementary information about the behaviour of winter temperatures. Then, the `grid.arrange()` function is used to display all the maps together for direct comparison.

In [ ]:
## seasonal analysis ##
obs.djf = subsetGrid(obs, ***)
getShape(obs.djf)

In [ ]:
## climatology ##
clim = climatology(***)

cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(10, 30, 1)
map.clim = spatialPlot(clim, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "DJF: Mean climatology (ºC)",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))

In [ ]:
## standard deviation ##
sdev = climatology(***, clim.fun = list(***))  # na.rm = TRUE ignores NAs in the computation of sd

cols = brewer.pal(9, "Reds")
cb.bins = seq(0, 5, 0.25)
map.sdev = spatialPlot(sdev, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "DJF: Standard deviation (ºC)",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))

In [ ]:
## yearly trends ##
obs.djf.y = aggregateGrid(obs.djf, ***)  # yearly aggregated data
trend.y = climatology(obs.djf.y, clim.fun = list(***))

cols = rev(brewer.pal(9, "RdBu"))
cb.bins = seq(-0.1, 0.1, 0.01)
map.trend.y = spatialPlot(***, 
            backdrop.theme = "countries",
            main = "DJF: Yearly trends (ºC/year)",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))

In [ ]:
## p98 ##
p98 = climatology(obs.djf, clim.fun = list(***))

cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(16, 36, 1)
map.p98 = spatialPlot(***, 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            main = "DJF: P98 (ºC)",
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))

In [ ]:
## plotting all the maps maps together ##
grid.arrange(map.clim, map.sdev, map.trend.y, map.p98)

## 8. Extending the Analysis to All Seasons

The previous section focused exclusively on winter (DJF) in order to illustrate the seasonal analysis workflow. However, climate studies rarely examine a single season in isolation. Instead, the same diagnostics are usually computed for all four climatological seasons to identify how the behaviour of the variable changes throughout the year.

Repeating the same block of code four times would be inefficient and difficult to maintain. A much better approach is to automate the analysis using a loop.

In this section, the four standard climatological seasons are defined:

* DJF: December, January and February
* MAM: March, April and May
* JJA: June, July and August
* SON: September, October and November

For each season, the notebook computes exactly the same diagnostics introduced previously:

* mean climatology
* standard deviation
* interannual trend
* the 98th percentile (P98)

Each result is stored in a list so that it can be visualised later without repeating any calculations.

In [ ]:
##########################################################
## repeating the analysis for all seasons (with a loop) ##
##########################################################

## seasons' definition ##
sea = list(c(12, 1, 2), 3:5, 6:8, 9:11)

## initializaing output lists ##
clim = vector(mode = "list", length = length(sea))
sdev = vector(mode = "list", length = length(sea))
p98 = vector(mode = "list", length = length(sea))
trend.y = vector(mode = "list", length = length(sea))

## looping along seasons ##
for (isea in 1:length(sea)) {
  obs.sea = subsetGrid(obs, ***)
  
  clim[[isea]] = climatology(***)
  sdev[[isea]] = climatology(***)
  p98[[isea]] = climatology(***)
  
  obs.sea.y = aggregateGrid(***)
  trend.y[[isea]] = climatology(***)
}

### 8.1. Computing Climatologies (for each season)

After computing the climatology for each season, the next step consists of comparing them.

Displaying the four seasonal climatologies together provides a comprehensive overview of the annual cycle of maximum temperature over Pakistan. Rather than inspecting four independent figures, arranging all maps within a single panel makes seasonal contrasts much easier to identify.

Notice that all maps share exactly the same colour scale: using identical colour limits ensures that differences between panels reflect actual variations in temperature rather than changes in the plotting scale.

In [ ]:
## clim maps ##
cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(12, 42, 2)
map.clim = spatialPlot(makeMultiGrid(***, skip.temporal.check = TRUE), 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(4, 1),
            main = "Climatology (ºC)",
            names.attr = sapply(1:length(sea), function(isea) {paste(substr(month.abb[sea[[isea]]], 1, 1), collapse = "")}),
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))
print(map.clim)

### 8.2. Computing the Standard Deviation (for each season)

While seasonal climatologies describe the average maximum temperature, they do not provide information about the day-to-day variability within each season.

The seasonal standard deviation quantifies the typical spread of daily maximum temperatures around their seasonal mean. Higher values indicate greater day-to-day variability, whereas lower values reflect more stable temperature conditions.

Comparing the seasonal standard deviation across stations and seasons helps identify periods and regions where temperature variability is enhanced or reduced. Such differences may be associated with the influence of synoptic weather systems, regional circulation patterns, or local geographic factors.

As in the previous figure, all seasonal maps are displayed using a common colour scale, allowing the magnitude of temperature variability to be compared consistently across seasons.

In [ ]:
## sdev maps ##
cols = brewer.pal(9, "Reds")
cb.bins = seq(0, 5, 0.25)
map.sdev = spatialPlot(makeMultiGrid(***, skip.temporal.check = TRUE), 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(4, 1),
            main = "Standard deviation (ºC)",
            names.attr = sapply(1:length(sea), function(isea) {paste(substr(month.abb[sea[[isea]]], 1, 1), collapse = "")}),
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))
print(map.sdev)

### 8.3. Computing Interannual Trends (for each season)

Seasonal climatologies describe the average state of the climate system, but they provide no information about long-term evolution.

By comparing seasonal trends, we can investigate whether warming (or cooling) occurs uniformly throughout the year or whether certain seasons experience stronger changes.

Many climate studies have shown that long-term warming is not necessarily homogeneous across seasons. Therefore, analysing seasonal trends separately often reveals patterns that remain hidden in annual averages.

Again, all seasonal trend maps are displayed using a common diverging colour scale centred around zero, allowing positive and negative trends to be compared consistently.

In [ ]:
## yearly trends maps ##
cols = rev(brewer.pal(9, "RdBu"))
cb.bins = seq(-0.2, 0.2, 0.02)
map.trend.y = spatialPlot(makeMultiGrid(***, skip.temporal.check = TRUE), 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(4, 1),
            main = "Yearly trends (ºC/year)",
            names.attr = sapply(1:length(sea), function(isea) {paste(substr(month.abb[sea[[isea]]], 1, 1), collapse = "")}),
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))
print(map.trend.y)

### 8.4. Computing the 98th Percentile (for each season)

Mean conditions are only one component of climate variability. However, in many environmental applications, extreme temperatures have a greater societal impact than average values.

Displaying the extreme percentile maps for the four seasons together allows us to investigate questions such as:

* Are extremely hot temperatures concentrated in the same regions throughout the year?
* Which seasons exhibit the greatest spatial variability?

In [ ]:
## p98 maps ##
cols = brewer.pal(9, "YlOrRd")
cb.bins = seq(15, 55, 2)
map.p98 = spatialPlot(makeMultiGrid(***, skip.temporal.check = TRUE), 
            xlim = lon, ylim = lat,
            backdrop.theme = "countries",
            as.table = TRUE,
            layout = c(4, 1),
            main = "98th percentile (ºC)",
            names.attr = sapply(1:length(sea), function(isea) {paste(substr(month.abb[sea[[isea]]], 1, 1), collapse = "")}),
            pch = 19, cex = 1.5,
            col.regions = cols,
            set.min = min(cb.bins), set.max = max(cb.bins),
            cuts = cb.bins,
            colorkey = list(right = list(fun = draw.colorkey, 
                                         args = list(key = list(at = cb.bins, 
                                                                col = colorRampPalette(cols), 
                                                                labels = list(at = cb.bins[seq(1, length(cb.bins), 2)],
                                                                              cex = 1,
                                                                              labels = cb.bins[seq(1, length(cb.bins), 2)]))))))
print(map.p98)

### 8.4. Visualizing All Together

The final figure of this section combines all seasonal diagnostics into a single summary panel.

In [ ]:
## complete figure ##
grid.arrange(map.clim, map.sdev, map.trend.y, map.p98)

## 9. From Spatial Fields to Individual Weather Stations

Spatial analyses provide an overview of regional climate patterns, but they do not reveal the detailed temporal behaviour observed at individual locations. This is what we do in the following sections. 

### 9.1. Temporal Aggregation

The original observations are available at daily resolution. For some applications, however, it is useful to reduce short-term variability by computing averages over longer periods.

In this section, two additional time-scales are analyzed:

* monthly means, highlighting the seasonal cycle;
* annual means, emphasising long-term climate variability.

In [ ]:
## monthly data
obs.m = aggregateGrid(obs, ***)
getShape(obs.m)
getTimeResolution(obs.m)
head(obs.m$Dates$start)
tail(obs.m$Dates$start)

In [ ]:
## yearly data
obs.y = aggregateGrid(obs, ***)
getShape(obs.y)
getTimeResolution(obs.y)
head(obs.y$Dates$start)
tail(obs.y$Dates$start)

### 9.2. Selecting a Representative Weather Station

To illustrate the analysis of station data, we focus on Karachi airport, one of the meteorological stations included in the observational dataset.

Rather than selecting the station manually by its position, we retrieve it using its metadata. 

Once the station has been identified, all subsequent analyses can be carried out using only its observations (function `subsetGrid()`).

The `temporalPlot()` function allows for visualizing time series. Different time-scales (e.g. daily, monthly and yearly) can be plotted in one single figure.

In [ ]:
## extracting data for Karachi ##
lon.karachi = 67.133 
lat.karachi = 24.9

obs.karachi = subsetGrid(obs, lonLim = ***, latLim = ***)
getShape(***)

## plotting daily time-series for Karachi ##
temporalPlot(***) 

In [ ]:
## monthly time-series for Karachi ##
obs.m.karachi = subsetGrid(obs.m, lonLim = ***, latLim = ***)
getShape(***)
temporalPlot(***)  

In [ ]:
## yearly time-series for Karachi ##
obs.y.karachi = subsetGrid(obs.y, ***)
getShape(***)
temporalPlot(***)  

In [ ]:
## daily, monthly and yearly time-series in one figure (for Karachi)
temporalPlot(***, ***, ***, lwd = c(1, 2, 3))

## 10. From Observations to Reanalysis Data

So far, the notebook has focused exclusively on observations collected at meteorological stations.

In many climate studies, however, observations are complemented by reanalysis datasets. Reanalyses combine millions of observations with numerical weather prediction models through a process known as data assimilation. The resulting products provide spatially continuous estimates of atmospheric variables (both at surface and at different vertical levels), even in regions where direct observations are sparse.

One of the most widely used reanalyses is ERA5, produced by the European Centre for Medium-Range Weather Forecasts (ECMWF).

In the remainder of the notebook, station observations will be compared with ERA5 maximum temperature fields.

**Note:** It is important to remember that reanalysis products are not direct observations. Instead, they represent the best physically consistent estimate of the atmospheric state obtained by combining observations and numerical models.

### 10.1 Loading ERA5 Data

To load the ERA5 data, the `loadGridData()` from the `loadeR` package is used.

As already mentioned, one of the strengths of the climate4R framework is that different climate datasets share a common data structure. Consequently, most functions used previously can also be applied to ERA5 without modification.

After loading the dataset, we inspect its temporal coverage, spatial resolution and metadata.

**Note:** Before comparing two different datasets, always verify:

* temporal coverage;
* spatial resolution;
* variable definition;
* units;
* missing values.

In [ ]:
#############################################
## comparison: ERA5 vs. local observations ##
#############################################

## loading tmax from ERA5, for a small region encompassing Karahi (1997-2016) ##
era5 = loadGridData(sprintf("%s/tmax.nc", dirdata.era5), 
                    var = "tmax", 
                    lonLim = c(66, 68), latLim = c(24, 26), 
                    years = 1997:2016)  
getShape(era5)
getTimeResolution(era5)
range(era5$Dates$start)

In [ ]:
getGrid(***)  # spatially continuous field (0.25º x 0.25º)

In [ ]:
## plotting ERA5 climatology (for tmax) ##
spatialPlot(***, 
            backdrop.theme = "countries", 
            main = "ERA5 (Tmax): Mean climatology (K)",
            color.theme = "YlOrRd")

In [ ]:
getGridUnits(***)

## 10.2. Harmonizing ERA5 and Local Observations

Because observational records often contain missing days, the notebook first identifies dates that are unavailable in the station dataset.

The ERA5 dataset is then restricted to the common temporal period shared by both datasets (`intersectGrid()` function).

This preprocessing step ensures that all subsequent comparisons are fair and that statistical differences are not introduced simply because the datasets contain different dates.

In [ ]:
## temporally alligning local observations and ERA5 ##
era5 = intersectGrid(obs, ***, type = ***, which.return = 2)
getShape(era5)

Also, whilst ERA5 maximum temperature is in K, the observations are in ºC. Thus, we convert ERA5 from K to ºC.

In [ ]:
## conversion from K to degC ##
era5 = gridArithmetics(era5, ***, operator = ***)
spatialPlot(climatology(***), 
            backdrop.theme = "countries", 
            main = "ERA5 (Tmax): Mean climatology (ºC)",
            color.theme = "YlOrRd")

Once both datasets have been harmonised, we can begin the actual comparison.

The first step consists of extracting the ERA5 grid cell corresponding to the location of Karachi airport.

Unlike the observational dataset, ERA5 is defined on a regular latitude-longitude grid. Therefore, instead of selecting a station identifier, we retrieve the grid cell whose geographical coordinates coincide with the station location.

The resulting time series can then be displayed together with the observations.

This first visual comparison already provides valuable information about the overall agreement between both datasets.

In [ ]:
## ERA5 vs. obs at Karachi ##
era5.karachi = subsetGrid(era5, ***)
temporalPlot(***, ***)  ## time series

Visual inspection of a time series provides qualitative information, but quantitative comparisons require additional diagnostics.

One useful approach is to visually compare the probability distributions of both datasets (use the `plot_pdfs()` function). Moreover, to complement the visual comparison, the notebook computes the Perkins Skill Score (PSS) using the `computePSS()` function. The PSS is a metric commonly used to evaluate the similarity between two probability distributions. Unlike traditional statistics such as the mean error or the correlation coefficient, the PSS evaluates the overlap between the complete distributions. A value close to one indicates that both datasets exhibit very similar distributions, whereas lower values reveal increasing discrepancies.

Distribution-based metrics are particularly useful when evaluating climate simulations because many applications depend on reproducing the entire range of observed values rather than simply matching the average.

Another powerful diagnostic for comparing two datasets is the quantile-quantile plot, commonly referred to as the Q-Q plot.
Rather than comparing observations chronologically, a Q-Q plot compares the empirical quantiles of two distributions.
If both datasets have identical statistical distributions, all points should lie close to the 1:1 reference (e.g. the diagonal) line. Departures from this line reveal systematic differences between the two datasets.

For example:

* a constant offset produces an approximately parallel displacement;
* differences in variability modify the slope;
* discrepancies affecting only extreme values appear near the ends of the curve.

Because of their diagnostic value, Q-Q plots are routinely used during the evaluation of climate simulations and bias-correction methods.

In [ ]:
## plotting the two PDFs together ##
plot_pdfs(***, ***, "OBS", "ERA5", 
          xlab = "Tmax (ºC)", ylab = "Density")

In [ ]:
## computing the PSS between the two PDFs ##
computePSS(obs.karachi$***, era5.karachi$***, bins = 100)

In [ ]:
## Q-Q plot ##
qqplot(***, ***, pch = 19, cex = 0.75,
       main = "Q-Q plot: Karachi", xlab = "OBS: tmax (ºC)", ylab = "ERA5: tmax (ºC)")
abline(a = 0, b = 1, col = "red", lwd = 2)
grid()

Next, we compute the Pearson correlation coefficient between observations and ERA5. Correlation measures the strength of the linear relationship between two variables.

Unlike the probability density function or the Q-Q plot, correlation focuses on temporal co-variability rather than the statistical distribution. A high correlation indicates that both datasets tend to vary synchronously through time (even if systematic biases are present). Conversely, two datasets may exhibit similar climatological means while showing poor temporal agreement.

This illustrates why several complementary diagnostics are required when evaluating climate datasets.

In [ ]:
## correlation analysis ##
cor(***, ***, 
    method = "pearson", use = "complete.obs")

Daily observations contain considerable short-term variability arising from individual weather events. When annual averages are considered instead, much of this variability disappears, allowing long-term climate fluctuations to emerge more clearly. For this reason, next we repeat the comparison between observations and ERA5 using annually aggregated maximum temperatures.

In [ ]:
## ERA vs. OBS at Karachi, based on yearly data ##
era5.y = aggregateGrid(era5, ***)
era5.y.karachi = subsetGrid(era5.y, ***)
temporalPlot(***, ***)
cor(obs.y.karachi$***, era5.y.karachi$***, 
    method = "pearson", use = "complete.obs")

## Summary

Throughout this notebook we have explored a complete introductory workflow for analysing observational climate data using the climate4R framework.

The main steps included are:

* loading observational datasets;
* understanding the structure of climate4R Grid objects;
* computing climatological statistics;
* analysing seasonal behaviour;
* estimating interannual trends;
* visualising spatial patterns;
* extracting and analysing individual weather stations;
* comparing observations with ERA5 reanalysis data;
* evaluating agreement using graphical and statistical diagnostics.

Although the examples focused on maximum temperature over Pakistan, exactly the same workflow can be adapted to many other climate variables, regions and datasets.

The concepts introduced here form the basis of more advanced analyses, including climate model evaluation, bias correction, statistical downscaling and climate change impact assessment.

## Further Exploration

The analyses presented in this notebook can be extended in several ways.

Some possible exercises are:

* Repeat the complete analysis for minimum temperature (`tmin`) or precipitation (`pr`).
* Compare additional stations representing different climatic regions.
* Compute additional validation metrics such as RMSE, MAE or bias.

These extensions illustrate how a relatively simple exploratory workflow can evolve into a complete climate data analysis project.